In [ ]:
# Install gwexpy with pinned versions of core dependencies for reproducibility on Colab

%pip install -q "gwexpy[all]" "gwpy<5.0.0" "numpy<2.0.0" "scipy<1.13.0" "astropy<7.0.0"

# SegmentTable Basics

> [!TIP]
> For a more comprehensive guide including visualization, GravitySpy integration, and advanced operations, see the [Table / Segment User Guide](intro_table.ipynb).

This tutorial introduces `SegmentTable`, a container for segment-keyed analysis in **GWexpy**. Unlike a simple `EventTable`, `SegmentTable` is designed to hold not just metadata about time intervals, but also their corresponding payload (like TimeSeries or ASDs) with lazy-loading support.

In [1]:
import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import numpy as np
from gwpy.segments import Segment
from gwexpy.table import SegmentTable

# 1. Create simple segments
segs = [Segment(0, 4), Segment(4, 8), Segment(8, 12)]
st = SegmentTable.from_segments(segs, label=["A", "B", "C"])
st

,span,label
0,"(0, 4)",A
1,"(4, 8)",B
2,"(8, 12)",C


## Delayed Loading with SegmentCell

You can add payload columns that are only loaded when needed. This is useful for handling large datasets.

In [2]:
def my_loader():
    # Simulate loading data
    print("Loading series...")
    from gwpy.timeseries import TimeSeries
    return TimeSeries(np.random.randn(128), sample_rate=32)

# Add a payload column with a loader (sequence of callables)
st.add_series_column("raw", loader=[my_loader]*len(st), kind="timeseries")
st

,span,label,raw
0,"(0, 4)",A,<lazy: timeseries>
1,"(4, 8)",B,<lazy: timeseries>
2,"(8, 12)",C,<lazy: timeseries>


## Row-wise Processing

`SegmentTable` provides an `apply()` method to process each row and collect results into new columns.

In [3]:
def process_row(row):
    span = row["span"]
    return {"duration": float(span[1] - span[0]), "valid": True}

st2 = st.apply(process_row)
st2.display()

,span,label,duration,valid,raw
0,"(0, 4)",A,4.0,True,<lazy: timeseries>
1,"(4, 8)",B,4.0,True,<lazy: timeseries>
2,"(8, 12)",C,4.0,True,<lazy: timeseries>


## Fetch and Conversion

You can explicitly load lazy cells with `fetch()` or `materialize()`. Converting to pandas gives you a standard DataFrame for meta columns.

In [4]:
st2.fetch()
df = st2.to_pandas()
df.head()

Loading series...
Loading series...
Loading series...


,span,label,duration,valid
0,"(0, 4)",A,4.0,True
1,"(4, 8)",B,4.0,True
2,"(8, 12)",C,4.0,True
